# Cross-Attention Head Training Example

In [ ]:
from __future__ import annotations

import os
from dataclasses import replace

os.environ["KERAS_BACKEND"] = "torch"

from pathlib import Path

import anndata as ad
import numpy as np

import crested
from crested import Genome

from laika import (
    DataModuleConfig,
    ExperimentConfig,
    FinetuneConfig,
    HeadOnlyConfig,
    ModelConfig,
    TrainerInitConfig,
    TrunkConfig,
    TrainingPlanConfig,
    Predictor,
    evaluate,
    plot_per_gene_correlations,
    plot_predicted_vs_true_for_genes,
    global_correlations,
    run_experiment,
    SequenceDataModule,
)

## Data Paths

In [ ]:
BASE_DATA_DIR    = Path("...")
ADATA_PATH       = BASE_DATA_DIR / "spatial/whb_38_embeddings.h5ad" # Anndata with counts and precomputed embeddings
GENOME_FASTA     = BASE_DATA_DIR / "dna/GRCm39.genome.fa"
GTF_PATH         = BASE_DATA_DIR / "dna/gencode.vM38.chr_patch_hapl_scaff.annotation.gtf"
TRAIN_GENES_LIST = BASE_DATA_DIR / "TRAIN/genes_train.txt" # List of genes to use for training
TEST_GENES_LIST  = BASE_DATA_DIR / "TRAIN/genes_test.txt" # List of genes to use for testing
BORZOI_MODEL     = "borzoiprime_mouse_rep0" # Pretrained Borzoi Prime model to use as trunk
SPATIAL_EMB_KEY  = "X_nichecompass"

RUN_DIR          = Path("...")
DEVICE           = "cuda"

WANDB_PROJECT    = "laika-example"
WANDB_RUN_NAME   = "example-run-1"

RUN_DIR.mkdir(parents=True, exist_ok=True)

## Configuration

In [ ]:
DATA_CONFIG = DataModuleConfig(
    gtf_path=GTF_PATH,
    val_frac=0.1,
    spatial_emb_key=SPATIAL_EMB_KEY,
    seed=42,
    max_stochastic_shift=128,
)

TRAINER_CONFIG = TrainerInitConfig(
    save_dir=RUN_DIR,
    device=DEVICE,
    wandb_project=WANDB_PROJECT,
    wandb_run_name=WANDB_RUN_NAME,
)

In [ ]:
TRAINING_PLAN = TrainingPlanConfig(
    run_head_only=True,
    head_only=HeadOnlyConfig(
        learning_rate=5e-4,
        weight_decay=1e-4,
        epochs=50,
        loss_fn="poisson",
        correlation_loss_lambda=0.3,
        gradient_accumulation_steps=4,
        clip_grad_norm=1.0,
        scheduler="cosine",
        warmup_epochs=5,
        cells_per_gene=4096,
        genes_per_batch=1,
        num_workers=4,
        patience=12,
        normalize_targets=False,
        gene_repeat_factor=4,
        gc_frequency_steps=50,
        empty_cache_frequency_steps=50,
        precompute_batch_size=4,
    ),
    run_finetune=True,
    finetune=FinetuneConfig(
        trunk_lr=5e-6,
        head_lr=5e-5,
        weight_decay=1e-4,
        use_lora=True,
        lora_rank=8,
        load_head_only_checkpoint=True,
        epochs=25,
        loss_fn="poisson",
        correlation_loss_lambda=0.3,
        gradient_accumulation_steps=4,
        clip_grad_norm=1.0,
        scheduler="cosine",
        warmup_epochs=3,
        cells_per_gene=1024,
        genes_per_batch=1,
        num_workers=0,
        patience=10,
        normalize_targets=False,
        gene_repeat_factor=2,
        gc_frequency_steps=50,
        empty_cache_frequency_steps=50,
    ),
)

In [ ]:
EXPERIMENT_CONFIG = ExperimentConfig(
    trunk=TrunkConfig(model_path=BORZOI_MODEL),
    data=DATA_CONFIG,
    model=ModelConfig(
        head="cross_attention",
        head_kwargs={
            "d_model": 256,
            "hidden_dim": 128,
            "dropout": 0.3,
            "output_activation": "linear",
            "spatial_emb_dropout": 0.1,
            "trunk_emb_dropout": 0.15,
        },
        pool_factor=2,
    ),
    trainer=TRAINER_CONFIG,
    training=TRAINING_PLAN,
)

In [ ]:
def load_gene_list(path: Path) -> list[str]:
    return [g.strip() for g in path.read_text().splitlines() if g.strip()]


def load_data():
    adata = ad.read_h5ad(ADATA_PATH)
    print(f"AnnData: {adata.n_obs} cells")
    print(f"Spatial embeddings: {adata.obsm[SPATIAL_EMB_KEY].shape}")

    genome = Genome(GENOME_FASTA, annotation=GTF_PATH)
    crested.register_genome(genome)

    train_genes = load_gene_list(TRAIN_GENES_LIST)
    test_genes = load_gene_list(TEST_GENES_LIST)
    print(f"Train genes: {len(train_genes)}, Test genes: {len(test_genes)}")

    return adata, genome, train_genes, test_genes


def train(adata, genome, train_genes):
    result = run_experiment(
        adata=adata,
        genome=genome,
        genes=train_genes,
        config=EXPERIMENT_CONFIG,
    )
    return result


def run_evaluation(adata, genome, test_genes, checkpoint_type="finetune"):
    eval_data_config = replace(
        DATA_CONFIG,
        val_genes=test_genes,
        max_stochastic_shift=0,
    )
    dm_test = SequenceDataModule.from_config(
        adata=adata,
        genome=genome,
        genes=test_genes,
        config=eval_data_config,
    )
    dm_test.setup()

    if checkpoint_type == "head_only":
        predictor = Predictor.from_head_only(
            run_dir=RUN_DIR,
            trunk=BORZOI_MODEL,
            data_module=dm_test,
            device=DEVICE,
        )
    else:  # finetune
        predictor = Predictor.from_finetune(
            run_dir=RUN_DIR,
            trunk=BORZOI_MODEL,
            data_module=dm_test,
            device=DEVICE,
        )

    eval_genes = dm_test.val_genes
    expression_gt = dm_test.get_expression_for_genes(eval_genes)

    results = evaluate(
        predictor=predictor,
        genes=eval_genes,
        expression_matrix=expression_gt,
        spatial_embeddings=dm_test.spatial_embeddings,
    )

    s = results.summary()
    print(f"\n{'Metric':<35} {'Value':>10}")
    print("-" * 50)
    for key in [
        "global_pearson", "global_spearman",
        "per_gene_pearson_median", "per_gene_spearman_median",
        "per_cell_pearson_median", "per_cell_spearman_median",
    ]:
        print(f"{key:<35} {s[key]:>10.4f}")
    print(f"{'per_gene_nan_count':<35} {s['per_gene_nan_count']:>10}")
    print(f"{'n_genes':<35} {s['n_genes']:>10}")
    print(f"{'n_cells':<35} {s['n_cells']:>10}")

    # Mean-predictor baseline
    gene_means = results.y_true.mean(axis=1, keepdims=True)
    y_baseline = np.broadcast_to(gene_means, results.y_true.shape).copy()
    baseline = global_correlations(results.y_true, y_baseline)

    print(f"\n{'':=<60}")
    print(f"{'Metric':<35} {'Baseline':>10} {'Model':>10}")
    print(f"{'':=<60}")
    print(f"{'Global Pearson':<35} {baseline['pearson']:>10.4f} {s['global_pearson']:>10.4f}")
    print(f"{'Global Spearman':<35} {baseline['spearman']:>10.4f} {s['global_spearman']:>10.4f}")
    print(f"{'Per-gene Pearson (median)':<35} {'0.0000':>10} {s['per_gene_pearson_median']:>10.4f}")

    # Per-gene bar chart
    plot_per_gene_correlations(
        results,
        RUN_DIR / f"test_per_gene_pearson_{checkpoint_type}.png",
        metric="pearson",
    )

    # Top 5 / bottom 5 predicted-vs-true scatter plots
    pearson = np.asarray(results.per_gene_pearson, dtype=np.float64)
    valid_idx = np.where(np.isfinite(pearson))[0]
    if valid_idx.size > 0:
        order_desc = valid_idx[np.argsort(pearson[valid_idx])[::-1]]
        order_asc = valid_idx[np.argsort(pearson[valid_idx])]
        top_idx = list(order_desc[:5])
        bottom_idx = [int(i) for i in order_asc[:5] if int(i) not in top_idx]
        selected_idx = top_idx + bottom_idx
        selected_genes = [results.gene_names[int(i)] for i in selected_idx]

        print("\nSelected genes for predicted-vs-true plot:")
        print("  Top 5 Pearson:", ", ".join(results.gene_names[int(i)] for i in top_idx))
        print("  Bottom 5 Pearson:", ", ".join(results.gene_names[int(i)] for i in bottom_idx))

        plot_predicted_vs_true_for_genes(
            results,
            RUN_DIR / f"test_pred_vs_true_top5_bottom5_{checkpoint_type}.png",
            genes=selected_genes,
            max_cols=5,
            independent_axes=True,
        )
        plot_predicted_vs_true_for_genes(
            results,
            RUN_DIR / f"test_pred_vs_true_top5_bottom5_shared_axes_{checkpoint_type}.png",
            genes=selected_genes,
            max_cols=5,
            independent_axes=False,
        )

    return results

In [ ]:
adata, genome, train_genes, test_genes = load_data()
print(f"\nExperimentConfig:\n{EXPERIMENT_CONFIG}")

## Training

In [ ]:
result = train(adata, genome, train_genes)

## Evaluation

In [ ]:
EVAL_HEAD_ONLY = False
EVAL_FINETUNE = True

if EVAL_HEAD_ONLY and EXPERIMENT_CONFIG.training.run_head_only:
    run_evaluation(adata, genome, test_genes, checkpoint_type="head_only")

if EVAL_FINETUNE and EXPERIMENT_CONFIG.training.run_finetune:
    run_evaluation(adata, genome, test_genes, checkpoint_type="finetune")